In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ["XLA_FLAGS"] = " ".join(
    [
        "--xla_cpu_enable_fast_math=true",  # relax FP semantics, ~10–30% on FP-heavy kernels
        "--xla_cpu_use_thunk_runtime=true",  # newer, lower per-op dispatch overhead
        "--xla_cpu_multi_thread_eigen=false",  # for tiny kernels, single-thread can be faster
    ]
)


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger("geometry.surface.surface_types").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("storage.google_cloud").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)
logger = logging.getLogger(__name__)

In [ ]:
import dataclasses
import time
import warnings
from collections.abc import Sequence
from typing import Literal

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from constellaration.geometry import surface_utils_desc

from coilstellaration import (
    coilset_utils,
    data_utils,
    metrics_utils_v2,
    paths,
    types,
)
from coilstellaration.benchmark import scoring
from coilstellaration.benchmark import types as benchmark_types
from coilstellaration.machine_learning import model_definition, utils

In [ ]:
@dataclasses.dataclass(frozen=True)
class PinnedRun:
    label: str
    checkpoint_ids: tuple[str, ...]
    model_type: Literal["mlp", "mlp_ensemble", "res_mlp", "res_mlp_ensemble"]

PINNED_RUNS = [
    PinnedRun(
        label="EnsembleMLP",
        checkpoint_ids=(
            # "DYYTj2Sox8VMBRpPMMyca7h",
            # "DkcB44BqnZxXVzzAzyQpxjL",
            "DATBU4QA5qKaPe8sDWnRTYs",
        ),
        model_type="mlp_ensemble",
    ),
    PinnedRun(
        label="MLP",
        checkpoint_ids=(
            # "D2HbzeYjo57Aif48z5T6axt",
            # "DBWrDsSS8QGRqXdjperF2Uf",
            "DadmEGxFuwhuYYCkijiBFoA",
        ),
        model_type="mlp",
    ),
    PinnedRun(
        label="EnsembleResMLP",
        checkpoint_ids=(
            # "DVEzMxFXt7mn8WCxRknx5YM",
            # "D5hSCAYRixTMzyvqW2Y6xaG",
            "DkQr5wvo5RvTit2SfdFXrnE",
        ),
        model_type="res_mlp_ensemble",
    ),
    PinnedRun(
        label="ResMLP",
        checkpoint_ids=(
            # "DTHUTPuNmjDPFspmsTx7Pj2",
            # "De3VR8KGEZCt7quiZcVAxQF",
            "DHJZTp7c73mBgVzonKHXZth",
        ),
        model_type="res_mlp",
    ),
]

In [ ]:
AGGREGATE_SUFFIXES = ("/min", "/mean", "/max", "/std")

METRICS_SCALAR_FIELDS = (
    "n_coils_per_half_period",
    "on_axis_average_magnetic_field",
    "minor_radius",
    "linking_current_consistency",
    "toroidal_flux",
)

METRICS_ARRAY_FIELDS = (
    "normalized_field_error",
    "local_quadratic_flux",
    "quadratic_flux",
    "coil_to_coil_min_distances",
    "coil_to_plasma_min_distances",
    "coil_lengths",
    "coil_curvatures",
    "coil_current_lengths",
    "coil_integrated_curvatures",
    "coil_torsions",
    "coil_arclength_variances",
    "coil_currents",
    "coil_linking_numbers",
)

NORMALIZED_PROP_SPECS = (
    ("coil_to_coil_min_distances", "normalized_coil_to_coil_min_distances", False),
    ("coil_to_plasma_min_distances", "normalized_coil_to_plasma_min_distances", False),
    ("coil_lengths", "normalized_coil_lengths", False),
    ("coil_curvatures", "normalized_coil_curvatures", True),
    ("coil_torsions", "normalized_coil_torsions", True),
)


def flatten_stats(
    arr: list | jax.Array | float | int,
) -> tuple[float, float, float, float]:
    if isinstance(arr, (int, float)):
        return float(arr), float(arr), float(arr), 0.0
    flat = np.asarray(arr, dtype=float).ravel()
    if flat.size == 0:
        return 0.0, 0.0, 0.0, 0.0
    return (
        float(flat.min()),
        float(flat.mean()),
        float(flat.max()),
        float(flat.std()),
    )


def parse_metrics_record(
    metrics: types.Metrics,
) -> dict:
    aggregates: dict = {}
    for field in METRICS_SCALAR_FIELDS:
        value = getattr(metrics, field)
        if value is None:
            continue
        aggregates[field] = float(getattr(metrics, field))

    for field in METRICS_ARRAY_FIELDS:
        value = getattr(metrics, field)
        if value is None:
            continue
        if isinstance(value, Sequence | jax.Array | np.ndarray):
            mn, mean, mx, std = flatten_stats(value)
            aggregates[f"{field}/min"] = mn
            aggregates[f"{field}/mean"] = mean
            aggregates[f"{field}/max"] = mx
            aggregates[f"{field}/std"] = std
        else:
            aggregates[field] = float(value)

    minor_radius = float(metrics.minor_radius)
    for raw_field, norm_field, multiply in NORMALIZED_PROP_SPECS:
        value = getattr(metrics, raw_field)
        if value is None or not isinstance(value, Sequence | jax.Array | np.ndarray):
            continue
        mn, mean, mx, std = flatten_stats(value)
        factor = minor_radius if multiply else 1.0 / minor_radius
        aggregates[f"{norm_field}/min"] = mn * factor
        aggregates[f"{norm_field}/mean"] = mean * factor
        aggregates[f"{norm_field}/max"] = mx * factor
        aggregates[f"{norm_field}/std"] = std * factor

    return aggregates

In [ ]:
def compute_data_row(eval_data: types.EvalData) -> dict[str, float | int | str]:
    with warnings.catch_warnings(action="ignore"):
        assert eval_data.predicted_coilset is not None
        boundary=surface_utils_desc.to_desc_fourier_rz_toroidal_surface(
                eval_data.boundary
            )
        predicted_metrics = metrics_utils_v2.evaluate_coilset_metrics_from_boundary(
            boundary=boundary,
            coilset=coilset_utils.coilstellaration_to_desc(
                eval_data.predicted_coilset
            ),
            subset_metrics=True,
        )
        # Can be very slow depending on HF, and local network speeds
        # true_metrics = data_utils.load_metrics_by_id(eval_data.true_metrics)
        true_metrics = metrics_utils_v2.evaluate_coilset_metrics_from_boundary(
            boundary=boundary,
            coilset=coilset_utils.coilstellaration_to_desc(
                eval_data.true_coilset
            ),
            subset_metrics=True,
        )


        metrics_utils_v2.evaluate_coilset_metrics_from_boundary(
            boundary=surface_utils_desc.to_desc_fourier_rz_toroidal_surface(
                eval_data.boundary
            ),
            coilset=coilset_utils.coilstellaration_to_desc(
                eval_data.true_coilset
            ),
            surf_eval_m=5,
            surf_eval_n=5,
            coil_eval_n=80,
            subset_metrics=True,
        )
        metrics_error = jax.tree_util.tree_map(
            lambda x, y: None if (x is None or y is None) else (x - y),
            predicted_metrics,
            true_metrics,
            is_leaf=lambda x: x is None,
        )
        metrics_error.surf_eval_coords = true_metrics.surf_eval_coords
        metrics_error.coil_eval_params -= true_metrics.coil_eval_params
        metrics_error.n_coils_per_half_period = true_metrics.n_coils_per_half_period
        metrics_error.minor_radius = true_metrics.minor_radius
        metrics_error_aggregates = parse_metrics_record(metrics_error)
        metrics_error_aggregates["boundary_id"] = eval_data.boundary_id

        score = scoring.score_instance(
            eval_data.boundary_id,
            data_utils.row_to_requirement_metrics(
                {
                    f"desc_metrics/{k}": v
                    for k, v in parse_metrics_record(predicted_metrics).items()
                }
            ),
            data_utils.row_to_requirement_metrics(
                {
                    f"desc_metrics/{k}": v
                    for k, v in parse_metrics_record(true_metrics).items()
                }
            ),
            benchmark_types.ScoringSettings(),
        )
        score_as_row = scoring.instance_score_to_row(score)
        score_as_row = {f"score/{k}": v for k, v in score_as_row.items()}

        return metrics_error_aggregates | score_as_row


def compute_data_dataframe(
    predictions: list[types.EvalData],
) -> pd.DataFrame:
    data_rows = []
    times = []
    for i, eval_data in enumerate(predictions):
        time_start = time.time_ns()
        data_row = compute_data_row(eval_data)
        time_end = time.time_ns()
        elapsed = time_end - time_start
        times.append(elapsed)
        data_rows.append(data_row)

        if i % 16 == 0:
            logger.info(f"Processed {i} / {len(predictions)}")
    logger.info(f"Finished processing all {len(predictions)} samples.")
    logger.info(f"Time for first iteration: {times[0] / 1e6:.2f} milliseconds")
    logger.info(
        f"Average processing time: {sum(times[1:]) / len(times[1:]) / 1e6:.2f} milliseconds"
    )

    return pd.DataFrame(data_rows)

In [ ]:
EVAL_SETTINGS = types.EvalSettings(
    # Keep low to speed up iteration during development; can increase for higher fidelity analysis.
    n_eval=8,
)

In [ ]:
eval_dataset = data_utils.load_benchmark_dataset(
    track="fixed_shape",
    stratum="tight",
    split="eval",
    n=EVAL_SETTINGS.n_eval,
)

In [ ]:
data_dfs = []
for pin in PINNED_RUNS:
    for checkpoint_id in pin.checkpoint_ids:
        logger.info("Evaluating pinned run %r seed=%s", pin.label, checkpoint_id)
        checkpoint = utils.read_checkpoint_from_id(pin.model_type, checkpoint_id)
        model = model_definition.read_model_from_checkpoint(checkpoint)
        logger.info(
            "Running predictions for pinned run %r seed=%s", pin.label, checkpoint_id
        )
        predictions = model_definition.predict_coilsets(
            model=model, eval_dataset=eval_dataset
        )
        logger.info(
            "Computing metrics errors for pinned run %r seed=%s",
            pin.label,
            checkpoint_id,
        )
        data_df = compute_data_dataframe(predictions)
        data_df["checkpoint_id"] = checkpoint_id
        data_df["model_type"] = pin.model_type
        data_df["label"] = pin.label

        feasibility = scoring.instance_score_to_row

        data_dfs.append(data_df)


In [ ]:
data_df = pd.concat(data_dfs, ignore_index=True)
data_df.to_parquet(paths.OUTPUTS_PATH / "data_df.parquet")

In [ ]:
columns_of_interest = [
    "normalized_field_error/mean",
    "normalized_coil_lengths/mean",
    "normalized_coil_to_coil_min_distances/mean",
    "normalized_coil_to_plasma_min_distances/mean",
    "normalized_coil_curvatures/mean",
]

In [ ]:
fig, ax = plt.subplots()
for column_of_interest in columns_of_interest:
    ax = sns.boxplot(
        data=data_df,
        x="label",
        y=column_of_interest,
    )
    plt.show()

In [ ]:
data_df.groupby("label")[
    [
        "normalized_field_error/mean",
        "normalized_coil_lengths/mean",
        "normalized_coil_to_coil_min_distances/mean",
        "normalized_coil_to_plasma_min_distances/mean",
        "normalized_coil_curvatures/mean",
    ]
].agg(["count", "mean", "std"]).T

In [ ]:
data_df.groupby("label")[[c for c in data_df.columns if c.startswith("score/")]].agg(
    ["max", "mean"]
).T

In [ ]:
sns.histplot(
    data=data_df,
    x="score/score",
    hue="label",
    bins=20,
    stat="count",
    multiple="dodge",
    common_norm=False,
)